## 1. Setup & Imports

In [1]:
import os
import sys
import yaml
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Add project root
sys.path.insert(0, '.')

print("✓ Imports ready")

2025-11-05 17:08:29.184485: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-05 17:08:29.194746: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762337309.207078  247596 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762337309.211214  247596 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762337309.221959  247596 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

✓ Imports ready


## 2. Load API Keys & Initialize

In [2]:
# Load API keys
api_keys_file = 'api_keys.yaml'
if not Path(api_keys_file).exists():
    api_keys_file = 'api_keys.example.yaml'

with open(api_keys_file, 'r') as f:
    api_keys = yaml.safe_load(f)

# Set environment variables
for key, value in api_keys.items():
    if value:
        os.environ[key] = str(value)

# Check Groq availability
has_groq = bool(api_keys.get('GROQ_API_KEY', '').startswith('gsk_'))

if has_groq:
    # Initialize Groq LLM
    try:
        from langchain_groq import ChatGroq
        llm = ChatGroq(
            model="llama-3.3-70b-versatile",
            api_key=os.environ['GROQ_API_KEY'],
            temperature=0.1,
            max_tokens=500,
        )
        print("✅ Groq LLM ready (llama-3.3-70b-versatile) - using langchain_groq")
    except ImportError:
        # Fallback to langchain_openai
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(
            model="llama-3.3-70b-versatile",
            api_key=os.environ['GROQ_API_KEY'],
            base_url='https://api.groq.com/openai/v1',
            temperature=0.1,
            max_tokens=500,
        )
        print("✅ Groq LLM ready (llama-3.3-70b-versatile) - using langchain_openai")
else:
    llm = None
    print("⚠️  Groq API key not found in api_keys.yaml")
    print("   Add GROQ_API_KEY to use LLM features")

✅ Groq LLM ready (llama-3.3-70b-versatile) - using langchain_groq


## 3. Load GO Graph Retriever

In [3]:
from query_engine.go_graph_retriever import GOGraphRetriever

# Load embedding model
print("Loading model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Initialize graph retriever
print("Loading graph retriever...")
graph_retriever = GOGraphRetriever(
    ontology_dir="data/kg/go/ontology",
    model=model
)

print("\n✅ GO Graph Retriever ready!")

Loading model...


/home/thuongnv/.pyenv/versions/3.11.9/lib/python3.11/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 803: system has unsupported display driver / cuda driver combination (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading graph retriever...
Loading hypergraph...
Loading graph indexes...
Loading graph indexes...
✓ Loaded 39,354 facts
✓ Loaded 286,288 hypernodes
✓ Loaded 39,354 GO term mappings
✓ Loaded 14,250 parent→children edges
✓ Loaded 39,351 child→parent edges


✅ GO Graph Retriever ready!
✓ Loaded 39,354 facts
✓ Loaded 286,288 hypernodes
✓ Loaded 39,354 GO term mappings
✓ Loaded 14,250 parent→children edges
✓ Loaded 39,351 child→parent edges


✅ GO Graph Retriever ready!


## 4. Smart Context Retrieval Function

In [4]:
def retrieve_with_smart_context(retriever, query, top_k=3, similarity_threshold=0.4, max_parents=2):
    """
    Smart retrieval for natural language queries
    
    Returns:
        List of enriched results with parent/child/part_of details
    """
    # Initial semantic retrieval
    initial_results = retriever.retrieve_semantic(query, top_k=top_k*2)
    query_emb = retriever.model.encode([query], normalize_embeddings=True)[0]
    
    enriched_results = []
    
    for result in initial_results[:top_k]:
        fact = result['fact']
        go_id = fact.get('GO id')
        if not go_id:
            continue
        
        # Get parents and filter by relevance
        parent_ids = retriever.get_parents(go_id)
        parent_details = []
        
        for parent_id in parent_ids:
            parent_fact = retriever.get_fact_by_id(parent_id)
            if not parent_fact:
                continue
            
            parent_text = f"{parent_fact.get('GO label', '')} {parent_fact.get('GO definition', '')}"
            parent_emb = retriever.model.encode([parent_text], normalize_embeddings=True)[0]
            similarity = float(np.dot(query_emb, parent_emb))
            
            if similarity >= similarity_threshold:
                parent_details.append({
                    'id': parent_id,
                    'label': parent_fact.get('GO label', 'N/A'),
                    'definition': parent_fact.get('GO definition', 'N/A'),
                    'namespace': parent_fact.get('GO namespace', 'N/A'),
                    'relevance_score': similarity
                })
        
        parent_details.sort(key=lambda x: x['relevance_score'], reverse=True)
        parent_details = parent_details[:max_parents]
        
        # Get children (top 3)
        child_ids = retriever.get_children(go_id)[:3]
        child_details = []
        for child_id in child_ids:
            child_fact = retriever.get_fact_by_id(child_id)
            if child_fact:
                child_details.append({
                    'id': child_id,
                    'label': child_fact.get('GO label', 'N/A')
                })
        
        # Get part_of chain
        part_of_chain = []
        rels = retriever.relationship_graph.get(go_id, {})
        part_of_ids = rels.get('part_of', [])
        
        for part_of_id in part_of_ids[:2]:
            part_of_fact = retriever.get_fact_by_id(part_of_id)
            if part_of_fact:
                part_of_chain.append({
                    'id': part_of_id,
                    'label': part_of_fact.get('GO label', 'N/A'),
                    'definition': part_of_fact.get('GO definition', 'N/A')
                })
        
        enriched_results.append({
            'go_id': go_id,
            'label': fact.get('GO label', 'N/A'),
            'definition': fact.get('GO definition', 'N/A'),
            'namespace': fact.get('GO namespace', 'N/A'),
            'score': result['score'],
            'parent_details': parent_details,
            'child_details': child_details,
            'part_of_details': part_of_chain
        })
    
    return enriched_results

print("✓ Function defined: retrieve_with_smart_context()")

✓ Function defined: retrieve_with_smart_context()


## 5. LLM Answer Generation Functions

In [5]:
def format_context_for_llm(results, method="smart"):
    """
    Format retrieval results into context for LLM
    
    Args:
        results: Retrieval results
        method: 'basic' or 'smart'
    """
    context_parts = []
    
    if method == "basic":
        # Basic format: chỉ có GO term info
        for i, r in enumerate(results, 1):
            fact = r['fact']
            context_parts.append(
                f"{i}. {fact.get('GO id')}: {fact.get('GO label')}\n"
                f"   Definition: {fact.get('GO definition')}\n"
                f"   Namespace: {fact.get('GO namespace')}\n"
            )
    else:
        # Smart format: include parent/child/part_of details
        for i, r in enumerate(results, 1):
            context_parts.append(
                f"{i}. {r['go_id']}: {r['label']}\n"
                f"   Definition: {r['definition']}\n"
                f"   Namespace: {r['namespace']}\n"
            )
            
            # Add parent details
            if r.get('parent_details'):
                context_parts.append("   Parent Terms (is_a relationships):\n")
                for p in r['parent_details']:
                    context_parts.append(
                        f"     - {p['id']}: {p['label']}\n"
                        f"       Definition: {p['definition']}\n"
                    )
            
            # Add part_of details
            if r.get('part_of_details'):
                context_parts.append("   Location (part_of relationships):\n")
                for loc in r['part_of_details']:
                    context_parts.append(
                        f"     - {loc['id']}: {loc['label']}\n"
                        f"       Definition: {loc['definition']}\n"
                    )
            
            context_parts.append("\n")
    
    return "".join(context_parts)


def answer_with_llm(question, context):
    """
    Generate answer using Groq LLM (via ChatOpenAI)
    """
    if not has_groq or llm is None:
        return "⚠️  Groq API key not configured. Add GROQ_API_KEY to api_keys.yaml"
    
    prompt = f"""You are an expert in Gene Ontology (GO) and molecular biology. Answer ONLY using the CONTEXT.

            CONTEXT:
            {context}

            QUESTION:
            {question}

            Rules:
            - The CONTEXT is your ONLY source of truth.
            - Use ontology relations (is_a, part_of, occurs_in, regulates, etc.) ONLY if they are explicitly shown in the CONTEXT (e.g., in “Parent Terms (is_a relationships)” or similar lines).
            - Do NOT invent or infer new relations or facts from the Definition text, outside knowledge, or common sense.
            - You may paraphrase Definitions but may NOT create new ontology relations from them.
            - If the CONTEXT is insufficient to answer, say: “I cannot answer this from the given context.”

            Now answer the QUESTION concisely, following these rules.

"""
    
    try:
        response = llm.invoke(prompt)
        return response.content
    except Exception as e:
        return f"❌ Error calling Groq API: {str(e)}"

print("✓ Functions defined: format_context_for_llm(), answer_with_llm()")

✓ Functions defined: format_context_for_llm(), answer_with_llm()


## 6. End-to-End Q&A Function

In [6]:
def qa_with_go(question, method="smart", show_context=False):
    """
    Complete Q&A pipeline: Retrieval → LLM → Answer
    
    Args:
        question: Natural language question
        method: 'basic' or 'smart'
        show_context: Show retrieved context
    """
    print("=" * 100)
    print(f"QUESTION: {question}")
    print("=" * 100)
    
    # Step 1: Retrieval
    print(f"\n🔍 Step 1: Retrieval ({method} method)")
    print("─" * 100)
    
    if method == "smart":
        results = retrieve_with_smart_context(
            graph_retriever,
            question,
            top_k=3,
            similarity_threshold=0.4,
            max_parents=2
        )
        print(f"Retrieved {len(results)} GO terms with enhanced context")
    else:
        results = graph_retriever.retrieve_semantic(question, top_k=3)
        print(f"Retrieved {len(results)} GO terms (basic)")
    
    # Step 2: Format context
    print(f"\n📝 Step 2: Formatting context for LLM")
    context = format_context_for_llm(results, method=method)
    context_size = len(context)
    print(f"Context size: {context_size:,} chars (~{context_size // 4} tokens)")
    
    if show_context:
        print("\n" + "─" * 100)
        print("CONTEXT:")
        print("─" * 100)
        print(context)
        print("─" * 100)
    
    # Step 3: LLM Answer
    print(f"\n🤖 Step 3: Generating answer with Groq LLM")
    print("─" * 100)
    answer = answer_with_llm(question, context)
    
    print("\n" + "=" * 100)
    print("📌 ANSWER:")
    print("=" * 100)
    print(answer)
    print("=" * 100)
    
    return {
        'question': question,
        'method': method,
        'results': results,
        'context': context,
        'context_size': context_size,
        'answer': answer
    }

print("✓ Function defined: qa_with_go()")

✓ Function defined: qa_with_go()


## 7. Quick Test: Single Question

**Purpose:** Test one question to verify setup is working

**Note:** Change `test_question` variable to test different queries

In [ ]:

test_question = "What is apoptosis?"

result = qa_with_go(
    test_question,
    method="smart",
    show_context=False  # Set True to see full context
)

# Show quick stats
print("\n" + "--" * 50)
print(f"✓ Retrieved {len(result['results'])} GO terms")
print(f"✓ Context size: {result['context_size']:,} chars")
print(f"✓ Answer length: {len(result['answer'])} chars")
print("--" * 50)

QUESTION: What is apoptosis?

🔍 Step 1: Retrieval (smart method)
────────────────────────────────────────────────────────────────────────────────────────────────────
Retrieved 3 GO terms with enhanced context

📝 Step 2: Formatting context for LLM
Context size: 2,618 chars (~654 tokens)

🤖 Step 3: Generating answer with Groq LLM
────────────────────────────────────────────────────────────────────────────────────────────────────

📌 ANSWER:
Apoptosis (GO:0006915) is a programmed cell death process (is_a GO:0012501) that involves a series of biochemical events leading to the death of a cell, characterized by distinct morphological changes, including the formation of apoptotic bodies (GO:0097189). It can be modulated by various processes, including suppression of apoptosis (GO:0033668), which can be mediated by symbionts.

📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊
✓ Retrieved 3 GO terms
✓ Context size: 2,618 chars
✓ Answer length: 386 chars
📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊

## 8. Compare: Basic vs Smart (DEMO)

**Purpose:** Show difference between Basic and Smart retrieval

**Note:** Only run this cell ONCE to see clean comparison

In [7]:

comparison_question = "“Mitochondrial inner membrane” chỉ là một loại màng (membrane) nói chung, hay nó còn được phân loại cụ thể hơn? Nó có được xem là một thành phần tế bào (cellular component) không?"

# BASIC method
result_basic = qa_with_go(comparison_question, method="basic", show_context=False)

print("\n" + "⏸-- " * 50 + "\n")

# SMART method  
result_smart = qa_with_go(comparison_question, method="smart", show_context=False)

# Quick comparison
print("\n" + "--" * 50)
print("COMPARISON:")
print(f"Basic:  {result_basic['context_size']:,} chars → Answer: {len(result_basic['answer'])} chars")
print(f"Smart:  {result_smart['context_size']:,} chars → Answer: {len(result_smart['answer'])} chars")
print("--" * 50)

QUESTION: “Mitochondrial inner membrane” chỉ là một loại màng (membrane) nói chung, hay nó còn được phân loại cụ thể hơn? Nó có được xem là một thành phần tế bào (cellular component) không?

🔍 Step 1: Retrieval (basic method)
────────────────────────────────────────────────────────────────────────────────────────────────────
Retrieved 3 GO terms (basic)

📝 Step 2: Formatting context for LLM
Context size: 559 chars (~139 tokens)

🤖 Step 3: Generating answer with Groq LLM
────────────────────────────────────────────────────────────────────────────────────────────────────

📌 ANSWER:
"Mitochondrial inner membrane" là một loại màng và được xem là một thành phần tế bào (cellular component), thuộc namespace "cellular_component".

⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- ⏸-- 

QUESTION: “Mitochondrial inner membrane” chỉ là một loại màng (me

### 8.1 View Context Details

Run cell này để xem context gửi cho LLM

In [8]:
# View context sent to LLM
if 'result_basic' in globals() and 'result_smart' in globals():
    print("=" * 100)
    print("BASIC CONTEXT (sent to LLM)")
    print("=" * 100)
    print(result_basic['context'])
    
    print("\n\n" + "=" * 100)
    print("SMART CONTEXT (sent to LLM)")
    print("=" * 100)
    print(result_smart['context'])
else:
    print("⚠️ Run Section 8 first!")

BASIC CONTEXT (sent to LLM)
1. GO:0005743: mitochondrial inner membrane
   Definition: The inner, i.e. lumen-facing, lipid bilayer of the mitochondrial envelope. It is highly folded to form cristae.
   Namespace: cellular_component
2. GO:0005741: mitochondrial outer membrane
   Definition: The outer, i.e. cytoplasm-facing, lipid bilayer of the mitochondrial envelope.
   Namespace: cellular_component
3. GO:0031966: mitochondrial membrane
   Definition: Either of the lipid bilayers that surround the mitochondrion and form the mitochondrial envelope.
   Namespace: cellular_component



SMART CONTEXT (sent to LLM)
1. GO:0005743: mitochondrial inner membrane
   Definition: The inner, i.e. lumen-facing, lipid bilayer of the mitochondrial envelope. It is highly folded to form cristae.
   Namespace: cellular_component
   Parent Terms (is_a relationships):
     - GO:0031966: mitochondrial membrane
       Definition: Either of the lipid bilayers that surround the mitochondrion and form the mitoc